In [1]:
from typing import List, Tuple

import pandas as pd
import polars as pl
from fastcore.xtras import Path

In [4]:
folder = Path.cwd().parent / 'src' / 'rfdatahub' / 'datasources' / 'arquivos' / 'saida'
folder.ls()

(#14) [Path('d:/code/rfdatahub/src/rfdatahub/datasources/arquivos/saida/aero.parquet.gzip'),Path('d:/code/rfdatahub/src/rfdatahub/datasources/arquivos/saida/estacoes.parquet.gzip'),Path('d:/code/rfdatahub/src/rfdatahub/datasources/arquivos/saida/log.parquet.gzip'),Path('d:/code/rfdatahub/src/rfdatahub/datasources/arquivos/saida/radcom.parquet.gzip'),Path('d:/code/rfdatahub/src/rfdatahub/datasources/arquivos/saida/radcom_raw.parquet.gzip'),Path('d:/code/rfdatahub/src/rfdatahub/datasources/arquivos/saida/Release.json'),Path('d:/code/rfdatahub/src/rfdatahub/datasources/arquivos/saida/smp.parquet.gzip'),Path('d:/code/rfdatahub/src/rfdatahub/datasources/arquivos/saida/smp_raw.parquet.gzip'),Path('d:/code/rfdatahub/src/rfdatahub/datasources/arquivos/saida/srd.parquet.gzip'),Path('d:/code/rfdatahub/src/rfdatahub/datasources/arquivos/saida/srd_raw.parquet.gzip'),Path('d:/code/rfdatahub/src/rfdatahub/datasources/arquivos/saida/stel.parquet.gzip'),Path('d:/code/rfdatahub/src/rfdatahub/datasource

In [ ]:
def read_df(name: str, dtype_backend='pyarrow')->pd.DataFrame:
    return pd.read_parquet(folder / f'{name}.parquet.gzip', dtype_backend=dtype_backend)

In [ ]:
def _validate_inputs(
    df_left: pd.DataFrame,
    df_right: pd.DataFrame,
    on: str,
    lat_col: str = 'Latitude',
    long_col: str = 'Longitude',
    suffixes: Tuple[str, str] = ('_x', '_y')
) -> None:
    """Validate input DataFrames meet pre-merge requirements.
    
    Args:
        df_left: Left DataFrame in merge operation
        df_right: Right DataFrame in merge operation
        on: Column name used for merging
        lat_col: Name of latitude column (default 'Latitude')
        long_col: Name of longitude column (default 'Longitude')
        suffixes: Suffixes used during merge (default ('_x', '_y'))
        
    Raises:
        ValueError: If any validation check fails
    """
    # Validate merge key exists
    for df, side in zip([df_left, df_right], ['left', 'right']):
        if on not in df.columns:
            raise ValueError(f"Merge key column '{on}' missing in {side} DataFrame")

    # Validate geographic columns exist
    for df, side in zip([df_left, df_right], ['left', 'right']):
        for col in [lat_col, long_col]:
            if col not in df.columns:
                raise ValueError(f"Geographic column '{col}' missing in {side} DataFrame")

    # Check for existing merge suffixes
    for df, side in zip([df_left, df_right], ['left', 'right']):
        suffix_clash = [
            col for col in df.columns 
            if col.endswith(suffixes)
        ]
        if suffix_clash:
            raise ValueError(
                f"{side} DataFrame contains columns with merge suffixes {suffixes}: "
                f"{suffix_clash}. Rename these columns before merging."
            )

    # Validate merge key dtype
    for df, side in zip([df_left, df_right], ['left', 'right']):
        if not pd.api.types.is_numeric_dtype(df[on]):
            raise ValueError(
                f"Merge key '{on}' in {side} DataFrame must be numeric. "
                f"Found dtype: {df[on].dtype}"
            )

In [ ]:
def _validate_columns(df: pd.DataFrame, required_cols: List[str]):
    missing = set(required_cols) - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    
def merge_on_frequency(
    df_left: pd.DataFrame,  # Left DataFrame
	df_right: pd.DataFrame,  # Right DataFrame
	on: str = 'Frequência',  # Column to use as merge key
	cols2merge: Tuple = ('Entidade', 'Fonte'),  # Colums to concatenate
) -> pd.DataFrame:  # Resulted merged dataframe
    lat_col: str = 'Latitude',
    long_col: str = 'Longitude',
    suffix: Tuple[str, str] = ('_x', '_y')):
    """Refactored version with clear phases."""
    # Phase 1: Preprocessing
    _validate_inputs(df_left, df_right, on=on)
    df_left, df_right = _cast_to_string(df_left), _cast_to_string(df_right)
    
    # Phase 2: Outer Merge
    df_merged = pd.merge(..., suffixes=('_left', '_right'))
    left_only = df_merged[df_merged['_merge'] == 'left_only']
    right_only = df_merged[df_merged['_merge'] == 'right_only']
    both = df_merged[df_merged['_merge'] == 'both']
    
    # Phase 3: Handle Disjoint Rows
    if both.empty:
        return pd.concat([left_only, right_only])
    
    # Phase 4: Calculate Distances (Vectorized)
    both = _calculate_distances(both, lat_col, long_col)
    
    # Phase 5: Split Close/Far Rows
    close_rows, far_left, far_right = _split_close_far_rows(both, max_dist=MAX_DIST)
    
    # Phase 6: Merge Columns (e.g., Entidade, Fonte)
    close_rows = _concatenate_columns(close_rows, cols2merge, suffixes=('_left', '_right'))
    
    # Phase 7: Combine All Segments
    return pd.concat([left_only, far_left, close_rows, right_only, far_right])